# Depression Detection using Machine Learning
## Reddit Dataset Analysis & Classification

This notebook implements a depression detection model using text analysis on Reddit data with preprocessing, feature extraction, and multiple classification algorithms.

In [ ]:
import pandas as pd
data=pd.read_csv("depression_dataset_reddit_cleaned.csv")
print(data.shape)
print(data.isnull().sum())
print(data['clean_text'].value_counts())


In [ ]:
#Removing duplicates
data=data.drop_duplicates("clean_text")
print(data.shape)

In [ ]:
#converting into text
data["clean_text"]=data["clean_text"].str.lower()
print(data.head())

In [ ]:
#removing links
import re
data["clean_text"]=data["clean_text"].apply(lambda x:re.sub(r'http\S+|www\S+','',x))
print(data.head())

In [ ]:
#Remove Tags
data["clean_text"]=data["clean_text"].apply(lambda x:re.sub(r'<.*?>','',x))


In [ ]:
#Remove Punctuation
#Removing Numbers
#Removing all except Alpha bits
import string
data['clean_text'] = data['clean_text'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
data["clean_text"] = data["clean_text"].apply(lambda x: re.sub(r'\d+', ' ', x))
data["clean_text"] = data["clean_text"].apply(lambda x: re.sub(r'[^a-zA-Z\s]', ' ', x))
data["clean_text"] = data["clean_text"].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
print(data.head())

In [ ]:
#Tokenization
import nltk
nltk.download('punkt_tab')
data["token"]=data["clean_text"].apply(lambda x:nltk.word_tokenize(x))
print(data["token"])

In [ ]:
#Removing Stopwords
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words=set(stopwords.words("english"))
data["token"]=data["token"].apply(lambda x:[word for word in x if word not in stop_words])
print(data["token"])

In [ ]:
#Lemonisation
from nltk.stem import WordNetLemmatizer
nltk.download("wordnet")
lemmatizer=WordNetLemmatizer()
data["token"]=data["token"].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
print(data["token"])

In [ ]:
data["clean_texts"]=data["token"].apply(lambda x:" ".join(x))
print(data["clean_texts"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_texts'],
    data['is_depression'],
    test_size=0.3,
    random_state=42,
    stratify=data['is_depression']
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)   # NO fit here!


In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train_tfidf, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_smote.value_counts())


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

# Train Logistic Regression
log_reg = LogisticRegression(
    max_iter=500,
    n_jobs=-1,
    random_state=42
)
log_reg.fit(X_train_tfidf, y_train)

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_tfidf, y_train)

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

print("All models trained successfully!")


In [ ]:
# Make predictions with all models
y_pred_lr = log_reg.predict(X_test_tfidf)
y_pred_rf = rf_model.predict(X_test_tfidf)
y_pred_nb = nb_model.predict(X_test_tfidf)

print("Predictions completed for all models!")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Evaluate all models
models = {
    'Logistic Regression': y_pred_lr,
    'Random Forest': y_pred_rf,
    'Naive Bayes': y_pred_nb
}

print("=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)

for model_name, predictions in models.items():
    print(f"\n{model_name}:")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
    print(f"Precision: {precision_score(y_test, predictions):.4f}")
    print(f"Recall: {recall_score(y_test, predictions):.4f}")
    print(f"F1-Score: {f1_score(y_test, predictions):.4f}")
    print("-" * 60)

print("\nDetailed Classification Report (Logistic Regression):")
print(classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix (Logistic Regression):")
print(confusion_matrix(y_test, y_pred_lr))


In [ ]:
text = input("Enter text to check: ")

# Preprocess the input text (same steps as training data)
text_cleaned = text.lower()
text_cleaned = re.sub(r'http\S+|www\S+','',text_cleaned)
text_cleaned = re.sub(r'<.*?>','',text_cleaned)
text_cleaned = text_cleaned.translate(str.maketrans('', '', string.punctuation))
text_cleaned = re.sub(r'\d+', ' ', text_cleaned)
text_cleaned = re.sub(r'[^a-zA-Z\s]', ' ', text_cleaned)
text_cleaned = re.sub(r'\s+', ' ', text_cleaned).strip()

tokenized_text = nltk.word_tokenize(text_cleaned)
tokenized_text = [word for word in tokenized_text if word not in stop_words]
lemmatized_text = [lemmatizer.lemmatize(word) for word in tokenized_text]
final_clean_text = " ".join(lemmatized_text)

# Transform the cleaned text using the trained TF-IDF vectorizer
user_tfidf = vectorizer.transform([final_clean_text])

# Make a prediction using the trained Random Forest model
prediction = log_reg.predict(user_tfidf)[0]

if prediction == 1:
    print("→ Prediction: DEPRESSED 🌧️")
else:
    print("→ Prediction: NOT DEPRESSED ☀️")